# C12-classical-models — Practice p22 — Solution


The proposal leaks test information by scaling the full dataset and by selecting from test accuracy. Accuracy also ignores asymmetric costs, and an uncalibrated SVM score is not a risk probability. Reserve a stratified test set untouched. Within development data, use nested CV or a train/validation split; every fold fits its scaler only on fold-training rows, tunes regularization, and evaluates on its validation rows. Logistic regression optimizes BCE and directly models probability. A linear SVM optimizes hinge loss and returns a signed margin; if retained, calibrate it on a disjoint calibration fold or inside nested CV. Choose the deployment threshold on validation probabilities by minimum expected asymmetric cost, then lock the pipeline, calibrator, and threshold before one test evaluation. Report a discrimination measure such as ROC-AUC or PR-AUC, held-out BCE and Brier score, and a reliability diagram or expected calibration error. Logistic regression is the natural first baseline because calibrated risk is the required output; compare a calibrated SVM under identical folds and budgets.


In [ ]:
import numpy as np

audit_protocol_p22 = {
    "leakage": ("full-dataset scaling", "test-set model selection"),
    "split": "stratified development/calibration/test with untouched test",
    "fold_pipeline": ("fit scaler on fold training", "fit model", "score fold validation"),
    "outputs": {"logistic": "modeled probability", "svm": "uncalibrated margin score"},
    "regularization_source": "inner validation folds",
    "threshold_rule": "minimize C_FP*(1-p) versus C_FN*p on validation probabilities",
    "discrimination_metrics": ("ROC-AUC", "PR-AUC"),
    "probability_checks": ("BCE", "Brier", "reliability diagram"),
    "svm_calibration": "disjoint calibration fold or nested cross-validation",
}
cost_fp_p22 = 1.0
cost_fn_p22 = 4.0
cost_optimal_threshold_p22 = cost_fp_p22 / (cost_fp_p22 + cost_fn_p22)


### Answer check


In [ ]:
assert set(audit_protocol_p22["leakage"]) == {"full-dataset scaling", "test-set model selection"}
assert audit_protocol_p22["outputs"]["logistic"] == "modeled probability"
assert audit_protocol_p22["outputs"]["svm"] == "uncalibrated margin score"
assert len(audit_protocol_p22["probability_checks"]) >= 2
assert "fold training" in audit_protocol_p22["fold_pipeline"][0]
assert np.isclose(cost_optimal_threshold_p22, 0.2, atol=1e-12, rtol=1e-10)
assert cost_fn_p22 > cost_fp_p22 and cost_optimal_threshold_p22 < 0.5
